## Preprocessing

In [ ]:
# --- Prepare tensors and model
# (loads CSVs, builds per-sample PyG Dataset with correct train/val/test split)
#
# Fixes vs v1:
#   1. sample_id_col initialised to None before the search loop; raises KeyError if not found
#      (was: NameError crash or silent pass if no matching column existed)
#   2. Train/val/test split is performed BEFORE computing normalisation statistics,
#      which are now derived from training rows only to prevent data leakage.
#   3. focal_alpha (computed dynamically from the positive rate in TRAIN data) is
#      now actually passed to FocalLoss (was hardcoded 0.1 in the loss constructor).
#   4. The same normalisation stats (means/stds) are saved so they can be applied
#      identically at inference time on new Grasshopper exports.

import json
import torch
import config
import pandas as pd
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from c21_surrogate_model_v4 import create_model, FocalLoss

edge_csv = "v5_edge_C12_S9999_D20260430"
node_csv = "v5_node_C12_S9999_D20260430"

node_csv_path = config.GH_DATA_PATH / f"{node_csv}.csv"
edge_csv_path = config.GH_DATA_PATH / f"{edge_csv}.csv"

nodes_df = pd.read_csv(node_csv_path)
edges_df = pd.read_csv(edge_csv_path)

data_inpection = False

# ============================================================================
# TOPOLOGY
# ============================================================================

# Load topology (edge_index) saved as a JSON list [[src...],[dst...]]
edge_index_path = Path(config.DATA_IO_PATH) / "edge_index.json"
if not edge_index_path.exists():
    raise FileNotFoundError(
        f"edge_index.json not found at {edge_index_path}. Provide a valid topology file."
    )
with open(edge_index_path, "r") as f:
    edge_index_list = json.load(f)
edge_index = torch.tensor(edge_index_list, dtype=torch.long)

if edge_index.ndim != 2 or edge_index.shape[0] != 2:
    raise ValueError(
        f"edge_index must have shape [2, num_edges], got {tuple(edge_index.shape)}"
    )

num_edges          = int(edge_index.shape[1])
expected_num_nodes = int(edge_index.max().item()) + 1
print(f"Topology: {num_edges} edges, expecting {expected_num_nodes} nodes per sample.")

# ============================================================================
# COLUMN VALIDATION
# ============================================================================

node_cols = ["x", "y", "z", "Tx", "Ty", "Tz", "Rx", "Ry", "Rz", "Fz"]
edge_cols = ["Area", "Length", "E", "Iy", "Iz", "J", "EA/L"]

missing_node_cols = [c for c in node_cols if c not in nodes_df.columns]
if missing_node_cols:
    raise KeyError(
        f"Missing required node columns: {missing_node_cols}. "
        f"Please provide these columns in {node_csv_path}."
    )

missing_edge_cols = [c for c in edge_cols if c not in edges_df.columns]
if missing_edge_cols:
    raise KeyError(
        f"Missing required edge columns: {missing_edge_cols}. "
        f"Please provide these columns in {edge_csv_path}."
    )

if "Utilization" not in edges_df.columns:
    raise KeyError(
        f"Missing required target column 'Utilization' in {edge_csv_path}."
    )

# ============================================================================
# SAMPLE ID DETECTION  (fix #1: initialise to None, raise if not found)
# ============================================================================

sample_id_col = None
for col in ("sample_id", "Sample_ID", "SampleId"):
    if col in nodes_df.columns and col in edges_df.columns:
        sample_id_col = col
        break

if sample_id_col is None:
    raise KeyError(
        "No sample ID column found in CSVs. "
        "Expected one of: 'sample_id', 'Sample_ID', 'SampleId'. "
        "Ensure both node and edge CSVs contain the same sample ID column."
    )

print(f"Sample ID column detected: '{sample_id_col}'")

# ============================================================================
# SAMPLE VALIDATION
# ============================================================================

node_groups = nodes_df.groupby(sample_id_col)
edge_groups = edges_df.groupby(sample_id_col)
samples = sorted(
    set(node_groups.groups.keys()).intersection(edge_groups.groups.keys())
)
if not samples:
    raise ValueError("No matching sample IDs between node and edge CSVs.")

print(f"Found {len(samples)} matching samples.")

for s in samples:
    n_count = len(node_groups.get_group(s))
    e_count = len(edge_groups.get_group(s))
    if n_count != expected_num_nodes:
        raise ValueError(
            f"Sample {s}: node count {n_count} != expected {expected_num_nodes}"
        )
    if e_count != num_edges:
        raise ValueError(
            f"Sample {s}: edge count {e_count} != expected {num_edges}"
        )

# ============================================================================
# TRAIN / VAL / TEST SPLIT  (fix #2: split BEFORE computing normalisation stats)
# ============================================================================

torch.manual_seed(42)
shuffled = torch.randperm(len(samples)).tolist()
train_size = int(0.8 * len(samples))
val_size   = int(0.1 * len(samples))

train_indices = shuffled[:train_size]
val_indices   = shuffled[train_size:train_size + val_size]
test_indices  = shuffled[train_size + val_size:]

train_samples = [samples[i] for i in train_indices]
val_samples   = [samples[i] for i in val_indices]
test_samples  = [samples[i] for i in test_indices]

print(f"Split: Train={len(train_samples)} | Val={len(val_samples)} | Test={len(test_samples)}")

# ============================================================================
# NORMALISATION  (fix #2: statistics from TRAIN rows only, no leakage)
# ============================================================================

train_nodes = nodes_df[nodes_df[sample_id_col].isin(train_samples)]
train_edges = edges_df[edges_df[sample_id_col].isin(train_samples)]

node_feature_means = train_nodes[node_cols].mean()
node_feature_stds  = train_nodes[node_cols].std(ddof=0).replace(0, 1.0)
edge_feature_means = train_edges[edge_cols].mean()
edge_feature_stds  = train_edges[edge_cols].std(ddof=0).replace(0, 1.0)

print("Normalisation statistics computed from training data only (z-score, clipped to ±5 sigma).")

# Save stats so the same transform can be applied at inference time on new GH exports.
# Load at inference with: stats = torch.load("norm_stats.pt")
norm_stats = {
    "node_means": node_feature_means.to_dict(),
    "node_stds":  node_feature_stds.to_dict(),
    "edge_means": edge_feature_means.to_dict(),
    "edge_stds":  edge_feature_stds.to_dict(),
    "node_cols":  node_cols,
    "edge_cols":  edge_cols,
}
norm_stats_path = Path(config.DATA_IO_PATH) / "norm_stats.pt"
torch.save(norm_stats, norm_stats_path)
print(f"Normalisation stats saved to {norm_stats_path}")

# ============================================================================
# CLASS BALANCE  (fix #3: compute pos_rate from TRAIN edges only)
# ============================================================================

train_pos_rate = float((train_edges["Utilization"] > 1).mean())
focal_alpha    = float(max(0.05, min(0.95, 1.0 - train_pos_rate)))
print(f"Train positive rate (Utilization>1): {train_pos_rate:.4f} -> focal_alpha={focal_alpha:.4f}")

# ============================================================================
# DATASET CONSTRUCTION
# ============================================================================

def build_sample(s, node_groups, edge_groups):
    """Build a single PyG Data object for sample s, applying train-derived normalisation."""
    n_df = node_groups.get_group(s)
    e_df = edge_groups.get_group(s)

    x = torch.tensor(
        ((n_df[node_cols] - node_feature_means) / node_feature_stds)
        .clip(-5, 5).values,
        dtype=torch.float32,
    )
    edge_attr = torch.tensor(
        ((e_df[edge_cols] - edge_feature_means) / edge_feature_stds)
        .clip(-5, 5).values,
        dtype=torch.float32,
    )
    y = torch.tensor(
        (e_df["Utilization"] > 1).astype(int).values,
        dtype=torch.float32,
    ).view(-1, 1)

    return Data(x=x, edge_index=edge_index.clone(), edge_attr=edge_attr, y=y)


train_dataset = [build_sample(s, node_groups, edge_groups) for s in train_samples]
val_dataset   = [build_sample(s, node_groups, edge_groups) for s in val_samples]
test_dataset  = [build_sample(s, node_groups, edge_groups) for s in test_samples]

print(
    f"Dataset constructed: {len(train_dataset)} train | "
    f"{len(val_dataset)} val | {len(test_dataset)} test samples. "
    f"Each sample: {expected_num_nodes} nodes, {num_edges} edges."
)

# ============================================================================
# MODEL
# ============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = create_model(
    node_features_dim=len(node_cols),
    edge_features_dim=len(edge_cols),
    device=device,
)
model.to(device)
print(
    f"Model on {device} | "
    f"node_features_dim={len(node_cols)} | edge_features_dim={len(edge_cols)}"
)

# ============================================================================
# DATALOADERS
# ============================================================================

batch_size = 32

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(
    f"DataLoaders ready — "
    f"Train: {len(train_dataloader)} batches | "
    f"Val: {len(val_dataloader)} batches | "
    f"Test: {len(test_dataloader)} batches "
    f"(batch_size={batch_size})"
)

# ============================================================================
# LOSS  (fix #3: use dynamically computed focal_alpha, not hardcoded 0.1)
# ============================================================================

# focal_alpha is derived above from the training-set positive rate.
# Using the data-driven value ensures the loss weighting matches your actual
# class balance rather than an arbitrary constant.
loss_fn = FocalLoss(alpha=focal_alpha, gamma=2.0)
print(f"FocalLoss(alpha={focal_alpha:.4f}, gamma=2.0)")

# ============================================================================
# DATA INSPECTION
# ============================================================================
if data_inpection:
    unsafe_per_sample = edges_df.groupby("Sample_ID").apply(
        lambda g: (g["Utilization"] > 1).sum()
    )
    print(unsafe_per_sample.describe())
    print(unsafe_per_sample.value_counts().sort_index().head(20))

    # in your preprocessing environment
    zero_unsafe = edges_df.groupby("Sample_ID").apply(
        lambda g: (g["Utilization"] > 1).sum() == 0
    )
    zero_ids = zero_unsafe[zero_unsafe].index
    print(edges_df[edges_df["Sample_ID"].isin(zero_ids)][edge_cols].describe())

# ============================================================================
# READY — objects available for training loop:
#   model, loss_fn, train_dataloader, val_dataloader, test_dataloader, device
#
# Recommended threshold at inference: 0.3-0.35 (not 0.5) to favour recall on
# the unsafe class. Tune on val set by inspecting confusion matrix + recall.
# ============================================================================

## Training

In [ ]:
# --- Training loop (follows c21_preprocessing_v2.py) ---
#
# Requires from preprocessing: model, loss_fn, train_dataloader, val_dataloader,
#                              test_dataloader, train_dataset, val_dataset,
#                              test_dataset, device, focal_alpha
#
# Improvements vs v1:
#   1. ReduceLROnPlateau scheduler — halves LR when val loss plateaus
#   2. Gradient clipping (max_norm=1.0) — prevents exploding gradients in NNConv
#   3. Early stopping with configurable patience — stops when val loss stops improving
#   4. Classification metrics at eval: recall, precision, F1, AUC-ROC, confusion matrix
#   5. Threshold sweep on val set to find the operating point that maximises recall
#      while keeping precision acceptable — result used for test evaluation
#   6. BCELoss option removed; FocalLoss with data-driven focal_alpha is always used
#   7. Checkpoint filename versioned to v4

import numpy as np
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

# ============================================================================
# HYPERPARAMETERS
# ============================================================================

EPOCHS          = 100
LR              = 1e-3
PATIENCE        = 20       # early stopping: stop if val loss doesn't improve for this many epochs
LR_FACTOR       = 0.5      # ReduceLROnPlateau: multiply LR by this on plateau
LR_PATIENCE     = 10       # ReduceLROnPlateau: epochs to wait before reducing LR
LR_MIN          = 1e-6     # minimum LR floor
GRAD_CLIP       = 1.0      # max gradient norm (set None to disable)
CKPT_PATH       = "surrogate_v4_checkpoint.pth"
focal_alpha     = 0.5 # from preprocessing, data-driven from train positive rate
experiment_mode = False

# Decision threshold: values below 0.5 bias toward catching unsafe members
# (false negatives are more dangerous than false positives in structural safety).
# We sweep the val set to find the best threshold, but this is the fallback.
DEFAULT_THRESHOLD = 0.35

# ============================================================================
# OPTIMIZER, SCHEDULER, LOSS
# ============================================================================

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=LR_MIN,
)

# focal_alpha comes from preprocessing (data-driven from train positive rate)
loss_fn = FocalLoss(alpha=focal_alpha, gamma=2.0)
print(f"FocalLoss(alpha={focal_alpha:.4f}, gamma=2.0)")

# ============================================================================
# EXPERIMENT
# ============================================================================
# iterate of different loss configurations and compare val AUC after 20 epochs

loss_functions = {
    "FocalLoss (data-driven)": FocalLoss(alpha=focal_alpha, gamma=2.0),
    "FocalLoss (moderate)":   FocalLoss(alpha=0.50, gamma=2.0),
    "BCELoss":                torch.nn.BCELoss(),
}

# ============================================================================
# TRAINING LOOP
# ============================================================================

train_losses  = []
val_losses    = []
epoch_history = []

best_val_loss = float("inf")
best_state    = None
best_epoch    = -1
epochs_no_improve = 0

print(f"\nStarting training: {EPOCHS} epochs, early stopping patience={PATIENCE}")
print("-" * 70)

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    epoch_train_loss = 0.0

    for batch in train_dataloader:
        batch = batch.to(device)
        optimizer.zero_grad()
        preds = model(batch.x, batch.edge_index, batch.edge_attr)
        loss  = loss_fn(preds, batch.y)
        loss.backward()

        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

        optimizer.step()
        epoch_train_loss += loss.item() * batch.num_graphs

    epoch_train_loss /= len(train_dataset)
    train_losses.append(epoch_train_loss)

    # ---- VALIDATE ----
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for batch in val_dataloader:
            batch = batch.to(device)
            preds = model(batch.x, batch.edge_index, batch.edge_attr)
            loss  = loss_fn(preds, batch.y)
            epoch_val_loss += loss.item() * batch.num_graphs

    epoch_val_loss /= len(val_dataset)
    val_losses.append(epoch_val_loss)

    # Step scheduler on val loss
    scheduler.step(epoch_val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # ---- CHECKPOINT ----
    if epoch_val_loss < best_val_loss:
        best_val_loss = float(epoch_val_loss)
        best_epoch    = int(epoch)
        best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    # ---- LOGGING (every 5 epochs) ----
    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:03d}  "
            f"train={epoch_train_loss:.6f}  "
            f"val={epoch_val_loss:.6f}  "
            f"lr={current_lr:.2e}  "
            f"no_improve={epochs_no_improve}/{PATIENCE}"
        )
        epoch_history.append((epoch + 1, epoch_train_loss, epoch_val_loss, current_lr))

    # ---- EARLY STOPPING ----
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch + 1} (no improvement for {PATIENCE} epochs).")
        break

print("-" * 70)

# ============================================================================
# RESTORE BEST CHECKPOINT
# ============================================================================

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best checkpoint from epoch {best_epoch + 1}  val_loss={best_val_loss:.6f}")
else:
    print("Warning: best_state was not set; using last epoch weights.")

# ============================================================================
# SAVE CHECKPOINT
# ============================================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_val_loss":    best_val_loss,
        "best_epoch":       best_epoch,
        "focal_alpha":      focal_alpha,
        "train_losses":     train_losses,
        "val_losses":       val_losses,
    },
    CKPT_PATH,
)
print(f"Checkpoint saved: {CKPT_PATH}")


# ============================================================================
# HELPER: collect predictions from a dataloader
# ============================================================================

def collect_preds(dataloader, model, device):
    """Returns (probs, targets) as flat numpy arrays."""
    model.eval()
    all_probs   = []
    all_targets = []
    with torch.no_grad():
        for batch in dataloader:
            batch = batch.to(device)
            probs = model(batch.x, batch.edge_index, batch.edge_attr)
            all_probs.append(probs.cpu())
            all_targets.append(batch.y.cpu())
    probs   = torch.cat(all_probs,   dim=0).view(-1).numpy()
    targets = torch.cat(all_targets, dim=0).view(-1).numpy()
    return probs, targets


def classification_report_at_threshold(probs, targets, threshold, label=""):
    """Print confusion matrix + classification report at a given threshold."""
    preds_binary = (probs >= threshold).astype(int)
    cm = confusion_matrix(targets.astype(int), preds_binary)
    print(f"\n{'='*60}")
    print(f"{label}  (threshold={threshold:.2f})")
    print(f"{'='*60}")
    print(f"Confusion matrix (rows=actual, cols=predicted):")
    print(f"              Pred Safe  Pred Unsafe")
    print(f"  Act Safe    {cm[0,0]:9d}  {cm[0,1]:11d}")
    print(f"  Act Unsafe  {cm[1,0]:9d}  {cm[1,1]:11d}")
    print()
    print(classification_report(
        targets.astype(int), preds_binary,
        target_names=["Safe (0)", "Unsafe (1)"],
        digits=4,
    ))
    recall_unsafe    = recall_score(targets, preds_binary, pos_label=1, zero_division=0)
    precision_unsafe = precision_score(targets, preds_binary, pos_label=1, zero_division=0)
    f1_unsafe        = f1_score(targets, preds_binary, pos_label=1, zero_division=0)
    print(f"Unsafe class  ->  Recall: {recall_unsafe:.4f}  Precision: {precision_unsafe:.4f}  F1: {f1_unsafe:.4f}")
    return recall_unsafe, precision_unsafe, f1_unsafe


# ============================================================================
# THRESHOLD SWEEP ON VALIDATION SET
# Finds the threshold that maximises recall on the unsafe class while keeping
# precision >= min_precision. For structural safety, catching failures matters
# more than avoiding false alarms.
# ============================================================================

print("\n--- Threshold sweep on validation set ---")
val_probs, val_targets = collect_preds(val_dataloader, model, device)

try:
    val_auc = roc_auc_score(val_targets, val_probs)
    print(f"Val AUC-ROC: {val_auc:.4f}")
except ValueError:
    print("Val AUC-ROC: n/a (only one class present in val targets)")
    val_auc = None

thresholds    = np.arange(0.10, 0.65, 0.05)
min_precision = 0.20   # lower bound: accept some false alarms to catch failures
best_threshold = DEFAULT_THRESHOLD
best_recall    = -1.0

sweep_results = []
for t in thresholds:
    preds_bin     = (val_probs >= t).astype(int)
    recall_u      = recall_score(val_targets, preds_bin, pos_label=1, zero_division=0)
    precision_u   = precision_score(val_targets, preds_bin, pos_label=1, zero_division=0)
    f1_u          = f1_score(val_targets, preds_bin, pos_label=1, zero_division=0)
    sweep_results.append((t, recall_u, precision_u, f1_u))
    if recall_u > best_recall and precision_u >= min_precision:
        best_recall    = recall_u
        best_threshold = t

print(f"\n{'Threshold':>10}  {'Recall(unsafe)':>15}  {'Precision(unsafe)':>18}  {'F1(unsafe)':>12}")
print("-" * 62)
for t, r, p, f in sweep_results:
    marker = " <-- selected" if abs(t - best_threshold) < 1e-6 else ""
    print(f"{t:10.2f}  {r:15.4f}  {p:18.4f}  {f:12.4f}{marker}")

print(f"\nSelected threshold: {best_threshold:.2f}  "
      f"(max recall >= {min_precision:.0%} precision constraint)")

# Full report on val set at chosen threshold
classification_report_at_threshold(val_probs, val_targets, best_threshold, label="VALIDATION SET")

# ============================================================================
# TEST SET EVALUATION
# ============================================================================

print("\n--- Test set evaluation ---")
test_probs, test_targets = collect_preds(test_dataloader, model, device)

try:
    test_auc = roc_auc_score(test_targets, test_probs)
    print(f"Test AUC-ROC: {test_auc:.4f}")
except ValueError:
    print("Test AUC-ROC: n/a (only one class present in test targets)")
    test_auc = None

# Evaluate at the threshold chosen on the val set
classification_report_at_threshold(test_probs, test_targets, best_threshold, label="TEST SET")

# Also show what 0.5 looks like for reference
classification_report_at_threshold(test_probs, test_targets, 0.5, label="TEST SET (default threshold=0.50, for reference)")

# ============================================================================
# COMPUTE AND PRINT TEST LOSS
# ============================================================================

model.eval()
test_loss = 0.0
with torch.no_grad():
    for batch in test_dataloader:
        batch = batch.to(device)
        preds = model(batch.x, batch.edge_index, batch.edge_attr)
        loss  = loss_fn(preds, batch.y)
        test_loss += loss.item() * batch.num_graphs
test_loss /= len(test_dataset)
print(f"\nTest focal loss: {test_loss:.6f}")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"  Best epoch:       {best_epoch + 1}")
print(f"  Best val loss:    {best_val_loss:.6f}")
print(f"  Test focal loss:  {test_loss:.6f}")
if val_auc:  print(f"  Val  AUC-ROC:     {val_auc:.4f}")
if test_auc: print(f"  Test AUC-ROC:     {test_auc:.4f}")
print(f"  Decision threshold (val-tuned): {best_threshold:.2f}")
print(f"  Checkpoint: {CKPT_PATH}")
print("=" * 70)

## Evaluation

In [ ]:
# =============================================================================
# Binary Classification Evaluation — TrussEdgeSafetyGNN
# =============================================================================
#
# Requires from the training script (c21_train_v2.py):
#   train_losses, val_losses      — per-epoch loss lists
#   test_probs, test_targets      — flat numpy arrays from collect_preds()
#   best_threshold                — threshold chosen on val set (from train script)
#   best_epoch                    — epoch at which best checkpoint was saved
#
# Produces:
#   Figure 1 — Training dynamics  (loss curves + LR schedule)
#   Figure 2 — Threshold analysis (ROC, PR, F1/Recall vs threshold)
#   Figure 3 — Prediction quality (probability distributions + calibration)
#   Figure 4 — Decision boundary  (confusion matrices at two thresholds)
#   Figure 5 — Per-member analysis (per-edge recall + hardest members)
#   metrics   — dict of all scalar metrics for export / reporting
#
# Fixes vs original:
#   - Duplicate block removed
#   - "BCE Loss" label corrected to "Focal Loss"
#   - Threshold selection: both max-F1 AND safety-oriented (max-recall @ P>=0.20)
#   - Misleading R²/MAE/RMSE aliases removed; named correctly
#   - Per-edge-ID recall breakdown added (unique to fixed-topology setting)
#   - Calibration curve added (reliability diagram)
#   - Matthews Correlation Coefficient added (best single metric for imbalanced binary)
#   - required_vars check uses only variables actually needed

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    accuracy_score,
    auc,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve

# =============================================================================
# 0. GUARD: check required variables exist
# =============================================================================

required_vars = [
    "train_losses", "val_losses",
    "test_probs", "test_targets",
    "best_threshold", "best_epoch",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(
        "Missing variables from training script: "
        + ", ".join(missing)
        + ". Run c21_train_v2.py first."
    )

NUM_EDGES = 120   # fixed topology — update if your truss changes

# =============================================================================
# 1. CORE METRICS
# =============================================================================

test_probs   = np.asarray(test_probs).flatten()
test_true    = np.asarray(test_targets).flatten().astype(int)
epochs_range = np.arange(1, len(train_losses) + 1)

# Curves
fpr, tpr, roc_thresholds        = roc_curve(test_true, test_probs)
roc_auc                          = roc_auc_score(test_true, test_probs)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(test_true, test_probs)
pr_auc                           = auc(recall_curve, precision_curve)

# Threshold A: maximise F1
f1_curve      = (2 * precision_curve * recall_curve) / (precision_curve + recall_curve + 1e-12)
idx_f1        = int(np.argmax(f1_curve[:-1]))   # exclude last point (undefined threshold)
threshold_f1  = float(pr_thresholds[idx_f1]) if len(pr_thresholds) > 0 else 0.5

# Threshold B: maximise recall on unsafe class subject to precision >= 20%
# (safety-oriented: missing a failure is worse than a false alarm)
MIN_PRECISION = 0.20
viable_mask   = precision_curve[:-1] >= MIN_PRECISION
if viable_mask.any():
    idx_safety       = int(np.argmax(recall_curve[:-1][viable_mask]))
    # map back to global index
    idx_safety       = np.where(viable_mask)[0][idx_safety]
    threshold_safety = float(pr_thresholds[idx_safety])
else:
    threshold_safety = threshold_f1   # fallback if precision is always < 20%

# Use val-set threshold from training script as the primary operating point,
# and show F1 and safety thresholds for comparison.
thr_primary  = float(best_threshold)   # from train script (val-tuned)

def scores_at(thr):
    pred = (test_probs >= thr).astype(int)
    cm   = confusion_matrix(test_true, pred)
    tn_, fp_, fn_, tp_ = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    return dict(
        threshold  = thr,
        pred       = pred,
        cm         = cm,
        tn=int(tn_), fp=int(fp_), fn=int(fn_), tp=int(tp_),
        accuracy   = accuracy_score(test_true, pred),
        precision  = precision_score(test_true, pred, zero_division=0),
        recall     = recall_score(test_true, pred, zero_division=0),
        f1         = f1_score(test_true, pred, zero_division=0),
        mcc        = matthews_corrcoef(test_true, pred),
    )

s_primary  = scores_at(thr_primary)
s_f1       = scores_at(threshold_f1)
s_safety   = scores_at(threshold_safety)
s_05       = scores_at(0.5)

brier = brier_score_loss(test_true, test_probs)

# =============================================================================
# 2. FIGURE 1 — Training Dynamics
# =============================================================================

fig1, axes = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle("Figure 1 — Training Dynamics", fontweight="bold", fontsize=13)

# Loss curves
ax = axes[0]
ax.plot(epochs_range, train_losses, "b-",  label="Train Loss", lw=2, marker="o", ms=3)
ax.plot(epochs_range, val_losses,   "r--", label="Val Loss",   lw=2, marker="s", ms=3)
ax.axvline(best_epoch + 1, color="green", linestyle=":", lw=1.5,
           label=f"Best epoch ({best_epoch + 1})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Focal Loss")
ax.set_title("Train vs Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Loss gap (overfitting indicator)
ax = axes[1]
gap = np.array(val_losses) - np.array(train_losses)
ax.plot(epochs_range, gap, color="purple", lw=2)
ax.axhline(0, color="black", linestyle="--", lw=1)
ax.axvline(best_epoch + 1, color="green", linestyle=":", lw=1.5,
           label=f"Best epoch ({best_epoch + 1})")
ax.fill_between(epochs_range, gap, 0,
                where=(gap > 0), alpha=0.2, color="red",   label="Overfitting region")
ax.fill_between(epochs_range, gap, 0,
                where=(gap < 0), alpha=0.2, color="green", label="Underfitting region")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val Loss − Train Loss")
ax.set_title("Generalisation Gap")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("eval_fig1_training_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================================
# 3. FIGURE 2 — Threshold Analysis (ROC, PR, metric vs threshold)
# =============================================================================

fig2, axes = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle("Figure 2 — Threshold Analysis", fontweight="bold", fontsize=13)

# ROC
ax = axes[0]
ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0, 1], [0, 1], "navy", lw=1.5, linestyle="--", label="Random")
# Mark operating points on ROC
for s, label, col in [(s_primary, f"val-tuned ({thr_primary:.2f})", "green"),
                       (s_f1,      f"max-F1 ({threshold_f1:.2f})",    "purple"),
                       (s_safety,  f"safety ({threshold_safety:.2f})", "red")]:
    ax.scatter(s["fp"] / max(s["fp"] + s["tn"], 1),
               s["tp"] / max(s["tp"] + s["fn"], 1),
               color=col, zorder=5, s=80, label=label)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)

# Precision-Recall
ax = axes[1]
ax.plot(recall_curve, precision_curve, color="teal", lw=2, label=f"PR AUC = {pr_auc:.3f}")
baseline = test_true.mean()
ax.axhline(baseline, color="gray", linestyle="--", lw=1.2, label=f"Baseline (pos rate={baseline:.2f})")
for s, label, col in [(s_primary, f"val-tuned", "green"),
                       (s_f1,      f"max-F1",    "purple"),
                       (s_safety,  f"safety",    "red")]:
    ax.scatter(s["recall"], s["precision"], color=col, zorder=5, s=80, label=label)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Metrics vs threshold sweep
ax = axes[2]
sweep_thresholds = np.linspace(0.05, 0.95, 100)
sweep_recall     = []
sweep_precision  = []
sweep_f1         = []
sweep_mcc        = []
for t in sweep_thresholds:
    pred = (test_probs >= t).astype(int)
    sweep_recall.append(recall_score(test_true, pred, zero_division=0))
    sweep_precision.append(precision_score(test_true, pred, zero_division=0))
    sweep_f1.append(f1_score(test_true, pred, zero_division=0))
    sweep_mcc.append(matthews_corrcoef(test_true, pred))

ax.plot(sweep_thresholds, sweep_recall,    "r-",  lw=2, label="Recall (unsafe)")
ax.plot(sweep_thresholds, sweep_precision, "b-",  lw=2, label="Precision (unsafe)")
ax.plot(sweep_thresholds, sweep_f1,        "g-",  lw=2, label="F1 (unsafe)")
ax.plot(sweep_thresholds, sweep_mcc,       "m--", lw=2, label="MCC")
ax.axvline(thr_primary,     color="green",  lw=1.5, linestyle=":", label=f"val-tuned ({thr_primary:.2f})")
ax.axvline(threshold_f1,    color="purple", lw=1.5, linestyle=":", label=f"max-F1 ({threshold_f1:.2f})")
ax.axvline(threshold_safety, color="red",  lw=1.5, linestyle=":", label=f"safety ({threshold_safety:.2f})")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("Metrics vs Threshold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("eval_fig2_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================================
# 4. FIGURE 3 — Prediction Quality & Calibration
# =============================================================================

fig3, axes = plt.subplots(1, 3, figsize=(18, 5))
fig3.suptitle("Figure 3 — Prediction Quality & Calibration", fontweight="bold", fontsize=13)

# Probability distribution by class
ax = axes[0]
ax.hist(test_probs[test_true == 0], bins=40, alpha=0.65,
        label=f"Safe (n={int((test_true==0).sum())})",   color="steelblue", edgecolor="white")
ax.hist(test_probs[test_true == 1], bins=40, alpha=0.65,
        label=f"Unsafe (n={int((test_true==1).sum())})", color="crimson",   edgecolor="white")
ax.axvline(thr_primary,     color="green",  lw=2, linestyle="--", label=f"val-tuned ({thr_primary:.2f})")
ax.axvline(threshold_safety, color="red",   lw=1.5, linestyle=":", label=f"safety ({threshold_safety:.2f})")
ax.axvline(0.5,              color="black", lw=1.2, linestyle="--", label="thr=0.50")
ax.set_xlabel("Predicted Probability P(safe)")
ax.set_ylabel("Count")
ax.set_title("Score Distribution by True Class")
# A good model shows two well-separated peaks; overlap = uncertainty region
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# Log-scale version (reveals low-probability tail behaviour)
ax = axes[1]
bins = np.linspace(0, 1, 41)
ax.hist(test_probs[test_true == 0], bins=bins, alpha=0.65,
        label="Safe",   color="steelblue", edgecolor="white")
ax.hist(test_probs[test_true == 1], bins=bins, alpha=0.65,
        label="Unsafe", color="crimson",   edgecolor="white")
ax.axvline(thr_primary, color="green", lw=2, linestyle="--", label=f"val-tuned ({thr_primary:.2f})")
ax.set_yscale("log")
ax.set_xlabel("Predicted Probability P(safe)")
ax.set_ylabel("Count (log scale)")
ax.set_title("Score Distribution (log scale)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# Calibration curve (reliability diagram)
# A perfectly calibrated model lies on the diagonal.
# Curves above diagonal = underconfident; below = overconfident.
ax = axes[2]
try:
    fraction_of_positives, mean_predicted = calibration_curve(
        test_true, test_probs, n_bins=10, strategy="uniform"
    )
    ax.plot(mean_predicted, fraction_of_positives,
            "bo-", lw=2, ms=6, label=f"GNN (Brier={brier:.3f})")
except Exception:
    ax.text(0.5, 0.5, "Calibration\nn/a", ha="center", va="center")
ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives (actual unsafe rate)")
ax.set_title("Calibration Curve (Reliability Diagram)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("eval_fig3_prediction_quality.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================================
# 5. FIGURE 4 — Confusion Matrices at Three Thresholds
# =============================================================================

fig4, axes = plt.subplots(1, 3, figsize=(16, 5))
fig4.suptitle("Figure 4 — Confusion Matrices", fontweight="bold", fontsize=13)

thresh_configs = [
    (s_05,      "Default (thr=0.50)"),
    (s_primary, f"Val-tuned (thr={thr_primary:.2f})"),
    (s_safety,  f"Safety-oriented (thr={threshold_safety:.2f})"),
]

for ax, (s, title) in zip(axes, thresh_configs):
    cm = s["cm"]
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Safe", "Unsafe"])
    ax.set_yticklabels(["Safe", "Unsafe"])
    for i in range(2):
        for j in range(2):
            val = cm[i, j]
            color = "white" if val > cm.max() * 0.6 else "black"
            ax.text(j, i, f"{val}", ha="center", va="center",
                    fontsize=13, fontweight="bold", color=color)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    # Annotate key metrics below each matrix
    ax.set_xlabel(
        f"Predicted\n"
        f"Recall(unsafe)={s['recall']:.3f}  Precision={s['precision']:.3f}\n"
        f"F1={s['f1']:.3f}  MCC={s['mcc']:.3f}  FN={s['fn']}",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig("eval_fig4_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================================
# 6. FIGURE 5 — Per-Member Analysis (fixed-topology only)
# =============================================================================
# Since every sample has exactly NUM_EDGES members in the same order, we can
# reshape predictions to [num_samples, NUM_EDGES] and compute per-member stats.
# This tells you WHICH structural members the model struggles with — highly
# relevant for a structural engineering thesis.

n_test_samples = len(test_probs) // NUM_EDGES

if len(test_probs) % NUM_EDGES == 0 and n_test_samples > 0:

    preds_mat   = (test_probs  >= thr_primary).astype(int).reshape(n_test_samples, NUM_EDGES)
    targets_mat = test_true.reshape(n_test_samples, NUM_EDGES)
    probs_mat   = test_probs.reshape(n_test_samples, NUM_EDGES)

    # Per-member stats
    per_member_unsafe_rate = targets_mat.mean(axis=0)           # true positive rate in dataset
    per_member_recall      = np.where(
        targets_mat.sum(axis=0) > 0,
        ((preds_mat == 1) & (targets_mat == 1)).sum(axis=0) / targets_mat.sum(axis=0).clip(1),
        np.nan,
    )
    per_member_fpr = np.where(
        (1 - targets_mat).sum(axis=0) > 0,
        ((preds_mat == 1) & (targets_mat == 0)).sum(axis=0) / (1 - targets_mat).sum(axis=0).clip(1),
        np.nan,
    )
    per_member_mean_prob = probs_mat.mean(axis=0)

    edge_ids = np.arange(NUM_EDGES)

    fig5, axes = plt.subplots(3, 1, figsize=(18, 12))
    fig5.suptitle("Figure 5 — Per-Member Analysis (fixed topology)", fontweight="bold", fontsize=13)

    # Plot A: unsafe rate per member (how often is this member critical?)
    ax = axes[0]
    colors_rate = ["crimson" if r > 0.3 else "steelblue" for r in per_member_unsafe_rate]
    ax.bar(edge_ids, per_member_unsafe_rate, color=colors_rate, edgecolor="none", width=0.8)
    ax.axhline(per_member_unsafe_rate.mean(), color="black", linestyle="--", lw=1.5,
               label=f"Mean unsafe rate = {per_member_unsafe_rate.mean():.3f}")
    ax.set_xlabel("Member ID")
    ax.set_ylabel("Fraction of samples where member is unsafe")
    ax.set_title("A — Unsafe Rate per Member  (red = >30% of samples)")
    ax.legend(fontsize=9)
    ax.set_xlim(-1, NUM_EDGES)
    ax.grid(True, alpha=0.2, axis="y")

    # Plot B: recall per member (how well does the model catch failures HERE?)
    ax = axes[1]
    recall_vals = np.nan_to_num(per_member_recall, nan=-0.05)
    colors_recall = []
    for r, rate in zip(recall_vals, per_member_unsafe_rate):
        if rate == 0:
            colors_recall.append("lightgray")   # member never fails — recall undefined
        elif r < 0.5:
            colors_recall.append("crimson")     # model misses >50% of failures here
        elif r < 0.8:
            colors_recall.append("orange")
        else:
            colors_recall.append("seagreen")
    ax.bar(edge_ids, recall_vals, color=colors_recall, edgecolor="none", width=0.8)
    ax.axhline(np.nanmean(per_member_recall), color="black", linestyle="--", lw=1.5,
               label=f"Mean recall = {np.nanmean(per_member_recall):.3f}")
    ax.axhline(0.8, color="seagreen", linestyle=":", lw=1.2, label="Target recall = 0.80")
    ax.set_xlabel("Member ID")
    ax.set_ylabel("Recall on unsafe class")
    ax.set_title("B — Per-Member Recall  (red<0.5, orange<0.8, green≥0.8, gray=never fails)")
    ax.set_ylim(-0.1, 1.05)
    ax.set_xlim(-1, NUM_EDGES)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis="y")

    # Plot C: mean predicted probability per member
    ax = axes[2]
    ax.bar(edge_ids, per_member_mean_prob, color="mediumpurple", edgecolor="none", width=0.8,
           label="Mean P(safe) predicted")
    ax.plot(edge_ids, per_member_unsafe_rate, "ro-", ms=3, lw=1.2,
            label="True unsafe rate (inverted reference)")
    ax.axhline(thr_primary, color="green", linestyle="--", lw=1.5,
               label=f"Decision threshold ({thr_primary:.2f})")
    ax.set_xlabel("Member ID")
    ax.set_ylabel("Mean predicted probability / unsafe rate")
    ax.set_title("C — Mean Predicted Probability vs True Unsafe Rate per Member")
    ax.set_xlim(-1, NUM_EDGES)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis="y")

    plt.tight_layout()
    plt.savefig("eval_fig5_per_member.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Identify the 10 hardest members (lowest recall among those that actually fail)
    members_that_fail = np.where(per_member_unsafe_rate > 0)[0]
    sorted_by_recall  = members_that_fail[np.argsort(per_member_recall[members_that_fail])]
    print("\nTop 10 hardest members (lowest recall among failing members):")
    print(f"  {'MemberID':>8}  {'UnsafeRate':>10}  {'Recall':>8}  {'FPR':>8}")
    for mid in sorted_by_recall[:10]:
        print(f"  {mid:>8d}  {per_member_unsafe_rate[mid]:>10.3f}  "
              f"{per_member_recall[mid]:>8.3f}  {per_member_fpr[mid]:>8.3f}")

else:
    print(f"\n[Per-member analysis skipped] "
          f"len(test_probs)={len(test_probs)} is not divisible by NUM_EDGES={NUM_EDGES}. "
          f"Check that test_probs comes from a contiguous evaluation of whole samples.")

# =============================================================================
# 7. METRICS SUMMARY — printed + exported dict
# =============================================================================

print("\n" + "=" * 65)
print("EVALUATION SUMMARY — TrussEdgeSafetyGNN")
print("=" * 65)
print(f"  Epochs trained:      {len(train_losses)}  (best: {best_epoch + 1})")
print(f"  Final train loss:    {train_losses[-1]:.6f}")
print(f"  Final val loss:      {val_losses[-1]:.6f}")
print()
print("  Threshold-independent:")
print(f"    ROC AUC:           {roc_auc:.4f}   (>0.90 = excellent, >0.80 = good)")
print(f"    PR  AUC:           {pr_auc:.4f}   (compare to baseline = {test_true.mean():.3f})")
print(f"    Brier score:       {brier:.4f}   (lower = better; 0 = perfect)")
print()
for label, s in [("Default (thr=0.50)", s_05),
                  (f"Val-tuned (thr={thr_primary:.2f})", s_primary),
                  (f"Safety   (thr={threshold_safety:.2f})", s_safety)]:
    print(f"  @ {label}:")
    print(f"    Accuracy:        {s['accuracy']:.4f}")
    print(f"    Precision:       {s['precision']:.4f}")
    print(f"    Recall (unsafe): {s['recall']:.4f}   ← most important for structural safety")
    print(f"    F1:              {s['f1']:.4f}")
    print(f"    MCC:             {s['mcc']:.4f}   (>0.5 = good for imbalanced data)")
    print(f"    TP={s['tp']}  TN={s['tn']}  FP={s['fp']}  FN={s['fn']}  "
          f"(FN = missed failures)")
    print()

print("  Saved figures:")
for i, name in enumerate([
    "eval_fig1_training_dynamics.png",
    "eval_fig2_threshold_analysis.png",
    "eval_fig3_prediction_quality.png",
    "eval_fig4_confusion_matrices.png",
    "eval_fig5_per_member.png",
], 1):
    print(f"    Fig {i}: {name}")
print("=" * 65)

# =============================================================================
# 8. METRICS DICT — for export / reporting cell
# =============================================================================

metrics = {
    # Threshold-independent
    "roc_auc":              float(roc_auc),
    "pr_auc":               float(pr_auc),
    "brier_score":          float(brier),
    # @ default threshold 0.5
    "acc_0.50":             float(s_05["accuracy"]),
    "precision_0.50":       float(s_05["precision"]),
    "recall_0.50":          float(s_05["recall"]),
    "f1_0.50":              float(s_05["f1"]),
    "mcc_0.50":             float(s_05["mcc"]),
    # @ val-tuned threshold
    "threshold_primary":    float(thr_primary),
    "acc_primary":          float(s_primary["accuracy"]),
    "precision_primary":    float(s_primary["precision"]),
    "recall_primary":       float(s_primary["recall"]),
    "f1_primary":           float(s_primary["f1"]),
    "mcc_primary":          float(s_primary["mcc"]),
    "tp_primary":           int(s_primary["tp"]),
    "tn_primary":           int(s_primary["tn"]),
    "fp_primary":           int(s_primary["fp"]),
    "fn_primary":           int(s_primary["fn"]),
    # @ safety-oriented threshold
    "threshold_safety":     float(threshold_safety),
    "recall_safety":        float(s_safety["recall"]),
    "precision_safety":     float(s_safety["precision"]),
    "f1_safety":            float(s_safety["f1"]),
    "mcc_safety":           float(s_safety["mcc"]),
    "fn_safety":            int(s_safety["fn"]),
    # Training
    "best_epoch":           int(best_epoch),
    "final_train_loss":     float(train_losses[-1]),
    "final_val_loss":       float(val_losses[-1]),
}

# EXPORT

In [ ]:
# Final evaluation export
import json, shutil
from datetime import datetime
from pathlib import Path
import torch
import numpy as np
import config

# Create artifact stem: ID{date}_LR{lr}_EP{epochs}_BS{bs}_FA{alpha}_ROC{roc:.3f}
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
artifact_stem = f"ID{ts}_LR{1e-3}_EP{epochs}_BS{batch_size}_FA{focal_alpha:.2f}_ROC{final_val_r2:.3f}"

# Export to config-defined directories (OneDrive)
models_dir = config.SM_EXPORT_PATH / artifact_stem
data_dir = config.SM_DATA_PATH / artifact_stem
models_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

# Save model checkpoint
ckpt_src = Path('surrogate_v3_checkpoint.pth')
model_target = models_dir / f"{artifact_stem}.pth"

if ckpt_src.exists():
    shutil.copy2(ckpt_src, model_target)
    print(f"Copied checkpoint to {model_target}")
else:
    torch.save({'model_state_dict': model.state_dict()}, model_target)
    print(f"Saved model state_dict to {model_target}")

# Save metrics JSON
metrics_path = data_dir / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

# Save figures (guarded)
try:
    training_visuals_fig.savefig(data_dir / f'{artifact_stem}_training.png', dpi=200)
except Exception as e:
    print(f'Failed to save training_visuals_fig: {e}')
try:
    pred_residuals_fig.savefig(data_dir / f'{artifact_stem}_roc_pr.png', dpi=200)
except Exception as e:
    print(f'Failed to save pred_residuals_fig: {e}')
try:
    error_dist_fig.savefig(data_dir / f'{artifact_stem}_prob_confmat.png', dpi=200)
except Exception as e:
    print(f'Failed to save error_dist_fig: {e}')

# Save raw predictions and targets
np.savetxt(data_dir / 'test_probs.csv', test_probs, delimiter=',')
np.savetxt(data_dir / 'test_true.csv', test_true, delimiter=',')

# Collect and display saved files
saved_files = sorted([str(p) for p in [model_target, metrics_path] + list(data_dir.glob('*'))])

print(f"\n✅ Evaluation exported to:")
print(f"   Models:  {models_dir}")
print(f"   Data:    {data_dir}")
print(f"\n📁 Files saved:")
for f in saved_files:
    print(f"   {f}")